In [6]:
import tkinter as tk
from tkinter import messagebox
import heapq
import random
import numpy as np

GRID = 6
CELL = 80

PLAYER_START = (0, 0)
ENEMY_START = (5, 0)
GOAL = (5, 5)

OBSTACLES = [(1,2),(2,2),(3,3)]

ACTIONS=[(-1,0),(1,0),(0,-1),(0,1)]

alpha=0.3
gamma=0.9
epsilon=1.0
epsilon_decay=0.995
epsilon_min=0.05

Q={}
running=False
episodes=0
wins=0
losses=0

player=list(PLAYER_START)
enemy=list(ENEMY_START)


def state():
    return (enemy[0],enemy[1],player[0],player[1])


def qvalues(s):
    if s not in Q:
        Q[s]=[0,0,0,0]
    return Q[s]


def heuristic(a,b):
    return abs(a[0]-b[0])+abs(a[1]-b[1])


def astar(start,goal):

    openlist=[(0,start)]
    came={}
    cost={start:0}

    while openlist:

        _,cur=heapq.heappop(openlist)

        if cur==goal:

            path=[]
            while cur in came:
                path.append(cur)
                cur=came[cur]

            path.append(start)
            return path[::-1]

        for dx,dy in ACTIONS:

            nxt=(cur[0]+dx,cur[1]+dy)

            if not(0<=nxt[0]<GRID and 0<=nxt[1]<GRID):
                continue

            if nxt in OBSTACLES:
                continue

            newcost=cost[cur]+1

            if nxt not in cost or newcost<cost[nxt]:

                cost[nxt]=newcost
                priority=newcost+heuristic(nxt,goal)
                heapq.heappush(openlist,(priority,nxt))
                came[nxt]=cur

    return []


root=tk.Tk()
root.title("A* + Q Learning")

canvas=tk.Canvas(root,width=GRID*CELL,height=GRID*CELL)
canvas.pack()

for i in range(GRID):
    for j in range(GRID):

        color="white"

        if (i,j) in OBSTACLES:
            color="black"

        elif (i,j)==GOAL:
            color="gold"

        canvas.create_rectangle(
            j*CELL,
            i*CELL,
            (j+1)*CELL,
            (i+1)*CELL,
            fill=color
        )

player_id=canvas.create_oval(20,20,60,60,fill="green")
enemy_id=canvas.create_oval(20,20,60,60,fill="red")

label=tk.Label(root,font=("Arial",12))
label.pack()


def update():

    canvas.coords(player_id,
                  player[1]*CELL+20,
                  player[0]*CELL+20,
                  player[1]*CELL+60,
                  player[0]*CELL+60)

    canvas.coords(enemy_id,
                  enemy[1]*CELL+20,
                  enemy[0]*CELL+20,
                  enemy[1]*CELL+60,
                  enemy[0]*CELL+60)

    label.config(text=f"Episodes:{episodes}   Wins:{wins}   Losses:{losses}   Epsilon:{epsilon:.2f}")


def reset():
    global player,enemy
    player=list(PLAYER_START)
    enemy=list(ENEMY_START)
    update()


def move_player():

    path=astar(tuple(player),GOAL)

    if len(path)>1:
        player[0],player[1]=path[1]


def move_enemy():

    global enemy,epsilon

    s=state()
    q=qvalues(s)

    if random.random()<epsilon:

        valid=[]

        for i,(dx,dy) in enumerate(ACTIONS):

            nx=enemy[0]+dx
            ny=enemy[1]+dy

            if 0<=nx<GRID and 0<=ny<GRID and (nx,ny) not in OBSTACLES:
                valid.append(i)

        action=random.choice(valid)

    else:

        m=max(q)
        best=[i for i,v in enumerate(q) if v==m]
        action=random.choice(best)

    dx,dy=ACTIONS[action]

    nx=enemy[0]+dx
    ny=enemy[1]+dy

    if not(0<=nx<GRID and 0<=ny<GRID):
        return

    if (nx,ny) in OBSTACLES:
        return

    olddist=heuristic(tuple(enemy),tuple(player))
    newdist=heuristic((nx,ny),tuple(player))

    reward=-1

    if newdist<olddist:
        reward=5

    if newdist>olddist:
        reward=-5

    if (nx,ny)==tuple(player):
        reward=100

    nextstate=(nx,ny,player[0],player[1])

    q[action]+=alpha*(reward+gamma*max(qvalues(nextstate))-q[action])

    enemy=[nx,ny]

    epsilon=max(epsilon_min,epsilon*epsilon_decay)


def loop():

    global running
    global episodes,wins,losses

    if not running:
        return

    move_player()
    move_enemy()
    update()

    if tuple(player)==GOAL:

        episodes+=1
        wins+=1
        messagebox.showinfo("Winner","Player reached Goal")
        reset()

    elif tuple(enemy)==tuple(player):

        episodes+=1
        losses+=1
        messagebox.showinfo("Enemy","Enemy caught Player")
        reset()

    root.after(250,loop)


def start():
    global running
    if not running:
        running=True
        loop()


def stop():
    global running
    running=False


def restart():
    global running
    running=False
    reset()


tk.Button(root,text="Start",bg="lightgreen",command=start).pack(fill="x")
tk.Button(root,text="Stop",bg="tomato",command=stop).pack(fill="x")
tk.Button(root,text="Restart",bg="skyblue",command=restart).pack(fill="x")

update()

root.mainloop()